# 📊 Ejercicios Prácticos: Transformaciones Avanzadas y Optimización de DataFrames

**Semana 2 - Gestión de Datos y Tecnologías Big Data**

---

## 🎯 Objetivos de Aprendizaje

Al completar este ejercicio práctico, serás capaz de:

1. ✅ Configurar Apache Spark en Google Colab
2. ✅ Cargar y transformar datos con DataFrames
3. ✅ Renombrar columnas y convertir tipos de datos
4. ✅ Aplicar Broadcast Joins para optimizar joins
5. ✅ Utilizar expresiones SQL avanzadas (expr, selectExpr, UDF)
6. ✅ Implementar cache y persistencia
7. ✅ Controlar particionamiento con repartition y coalesce
8. ✅ Trabajar con múltiples fuentes de datos (CSV, JSON)

---

## 📝 Instrucciones

- **Lee los comentarios** en cada línea de código antes de ejecutar
- **Ejecuta las celdas en orden** de arriba hacia abajo
- **Observa los resultados** y compáralos con las explicaciones
- **Experimenta** modificando parámetros para profundizar tu aprendizaje

---

⏱️ **Tiempo estimado:** 15 minutos


---
# PARTE 1: Configuración del Entorno Spark
---

En esta primera parte configuraremos todo el entorno necesario para trabajar con Apache Spark en Google Colab.

## 1.1 Montar Google Drive

Necesitamos acceder a los archivos de datos almacenados en Google Drive.

In [ ]:
# Importar la librería para interactuar con Google Drive
from google.colab import drive

# Montar Google Drive en la ruta /content/drive
# Te pedirá autorización para acceder a tu Drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1.2 Instalar Apache Spark 3.5.9

Instalamos Apache Spark y todas sus dependencias necesarias.

In [10]:
# Instalar Java 8 (es un requisito obligatorio para ejecutar Spark)
# -qq: modo silencioso, > /dev/null: oculta la salida
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# Eliminar cualquier archivo .tgz de Spark previamente descargado o incompleto
!rm -f spark-3.5.9-bin-hadoop3.tgz

# Descargar Apache Spark versión 3.5.9 con soporte para Hadoop 3
# Sin -q para ver el progreso y posibles errores de descarga
!wget https://downloads.apache.org/spark/spark-3.5.9/spark-3.5.9-bin-hadoop3.tgz

# Descomprimir el archivo descargado
# xf: extraer archivo (x=extract, f=file)
!tar xf spark-3.5.9-bin-hadoop3.tgz

# Instalar PySpark (interfaz de Python para Spark)
# Instalar findspark (ayuda a localizar Spark en el sistema)
# -q: modo silencioso
!pip install -q pyspark findspark

# Mensaje de confirmación
print("✅ Apache Spark 3.5.9 instalado correctamente")

--2026-07-20 13:08:10--  https://downloads.apache.org/spark/spark-3.5.9/spark-3.5.9-bin-hadoop3.tgz
Resolving downloads.apache.org (downloads.apache.org)... 88.99.208.237, 95.216.224.44, 2a01:4f9:2b:1cc2::2, ...
Connecting to downloads.apache.org (downloads.apache.org)|88.99.208.237|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 401379923 (383M) [application/x-gzip]
Saving to: ‘spark-3.5.9-bin-hadoop3.tgz’

spark-3.5.9-bin-had 100%[===================>] 382.79M  17.3MB/s    in 25s     

2026-07-20 13:08:36 (15.4 MB/s) - ‘spark-3.5.9-bin-hadoop3.tgz’ saved [401379923/401379923]

✅ Apache Spark 3.5.9 instalado correctamente


## 1.3 Configurar Variables de Entorno

Configuramos las variables de entorno para que Python y Spark se comuniquen correctamente.

In [11]:
# Importar librería para manipular variables del sistema operativo
import os

# JAVA_HOME: indica dónde está instalado Java
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# SPARK_HOME: indica dónde está instalado Spark
os.environ["SPARK_HOME"] = "/content/spark-3.5.9-bin-hadoop3"

# PYTHON_PATH: indica dónde están las librerías de Python para Spark
# py4j permite que Python y Java se comuniquen
os.environ["PYTHON_PATH"] = "/content/spark-3.5.9-bin-hadoop3/python/lib/py4j-0.10.9.7-src.zip"

# Mensaje de confirmación
print("✅ Variables de entorno configuradas para Spark 3.5.9")

✅ Variables de entorno configuradas para Spark 3.5.9


## 1.4 Crear Sesión de Spark

La sesión de Spark es el punto de entrada para trabajar con DataFrames y RDDs.

In [12]:
# Importar findspark para inicializar Spark
import findspark

# Inicializar findspark (busca y configura Spark automáticamente)
findspark.init()

# Importar SparkSession (clase principal para trabajar con Spark)
from pyspark.sql import SparkSession

# Crear una sesión de Spark con configuración personalizada
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('Manejo dataframe') \
        .getOrCreate()
# .master("local[*]"): ejecutar Spark localmente usando todos los cores disponibles
# .appName('Manejo dataframe'): nombre de la aplicación (aparece en la UI de Spark)
# .getOrCreate(): obtener sesión existente o crear una nueva si no existe

# Obtener el contexto de Spark (nivel más bajo de la API)
sc = spark.sparkContext

# Mostrar información de la sesión creada
print("✅ Sesión de Spark creada exitosamente")
print(f"📊 Versión de Spark: {spark.version}")

# Mostrar el SparkContext (incluye enlace a la UI de Spark)
sc

✅ Sesión de Spark creada exitosamente
📊 Versión de Spark: 3.5.9


<SparkContext master=local[*] appName=Manejo dataframe>

---
# PARTE 2: Características de DataFrames con Dataset Titanic
---

En esta parte aprenderemos a cargar, transformar y manipular DataFrames usando el famoso dataset del Titanic.

## 2.1 Cargar Archivo CSV

Veremos diferentes formas de cargar archivos CSV en Spark.

### 2.1.1 Cargar Archivo sin Formato

**Sintaxis:** `spark.read.csv('ruta_archivo_csv')`

Primero veremos qué pasa cuando cargamos un CSV sin especificar opciones.

In [13]:
# Cargar archivo CSV sin opciones (SIN header=True, SIN inferSchema=True)
# Resultado: Spark no sabe que la primera fila son encabezados
# Resultado: Todas las columnas serán de tipo string
# Resultado: Los nombres de columnas serán genéricos: _c0, _c1, _c2, etc.
df = spark.read.csv('Titanic.csv')

# NOTA: Debes tener el archivo Titanic.csv en tu Google Drive en esta ruta

In [14]:
# Mostrar las primeras 20 filas del DataFrame
# Observa que la primera fila contiene los nombres de las columnas originales
# Observa que las columnas se llaman _c0, _c1, _c2, etc.
df.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|        _c0|     _c1|   _c2|                 _c3|   _c4| _c5|  _c6|  _c7|             _c8|    _c9| _c10|    _c11|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
|          1|       0|     3|Braund, Mr. Owen ...|  male|  22|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|  38|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|  26|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|  35|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|  35|    0|    0|      

In [15]:
# Contar el número total de filas en el DataFrame
# Esta es una ACCIÓN (no una transformación)
# Resultado esperado: 892 filas (891 pasajeros + 1 fila de encabezados)
df.count()

892

### 2.1.2 Renombrar Columnas

**Sintaxis:** `df.withColumnRenamed("nombre_antiguo", "nombre_nuevo")`

Ahora renombraremos las columnas genéricas (_c0, _c1...) por nombres significativos.

In [16]:
# Renombrar todas las columnas del DataFrame
# withColumnRenamed() crea un NUEVO DataFrame (los DataFrames son inmutables)
# Podemos encadenar múltiples withColumnRenamed() con \
df2 = df.withColumnRenamed("_c0", "PassengerId") \
        .withColumnRenamed("_c1", "Survived") \
        .withColumnRenamed("_c2", "Pclass") \
        .withColumnRenamed("_c3", "Name") \
        .withColumnRenamed("_c4", "Sex") \
        .withColumnRenamed("_c5", "Age") \
        .withColumnRenamed("_c6", "SibSp") \
        .withColumnRenamed("_c7", "Parch") \
        .withColumnRenamed("_c8", "Ticket") \
        .withColumnRenamed("_c9", "Fare") \
        .withColumnRenamed("_c10", "Cabin") \
        .withColumnRenamed("_c11", "Embarked")

# Ahora df2 tiene nombres de columnas significativos

In [17]:
# Mostrar el DataFrame con las columnas renombradas
# Ahora los nombres de columnas son legibles
df2.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
|          1|       0|     3|Braund, Mr. Owen ...|  male|  22|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|  38|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|  26|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|  35|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|  35|    0|    0|      

In [18]:
# Contar filas del nuevo DataFrame
# Sigue siendo 892 (incluyendo la fila de encabezados originales)
df2.count()

892

### 2.1.3 Filtrar Filas del DataFrame

**Sintaxis:** `df.filter(df.columna condicion)`

Ahora eliminaremos la primera fila que contiene los encabezados originales.

In [19]:
# Filtrar para eliminar la fila que contiene "PassengerId" en la columna PassengerId
# != significa "diferente de"
# Esta es una TRANSFORMACIÓN (perezosa, no se ejecuta hasta una acción)
df = df2.filter(df2.PassengerId != "PassengerId")

# Ahora df solo contiene datos de pasajeros (sin la fila de encabezados)

In [20]:
# Mostrar el DataFrame filtrado
# Observa que ya no aparece la fila con "PassengerId", "Survived", etc.
df.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|  22|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|  38|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|  26|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|  35|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|  35|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

### 2.1.4 Ver Estructura del DataFrame

**Sintaxis:** `df.printSchema()`

Inspeccionamos el esquema (schema) para ver los tipos de datos de cada columna.

In [21]:
# Mostrar el esquema del DataFrame
# El esquema incluye: nombre de columna, tipo de dato, si acepta nulos
# Observa que TODAS las columnas son de tipo 'string'
# Esto es incorrecto para columnas numéricas como Age, Fare, etc.
df.printSchema()

root
 |-- PassengerId: string (nullable = true)
 |-- Survived: string (nullable = true)
 |-- Pclass: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- SibSp: string (nullable = true)
 |-- Parch: string (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: string (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



### 2.1.5 Modificar Tipo de Datos de Columnas

**Sintaxis:** `df.withColumn("NombreColumna", df.NombreColumna.cast("tipo"))`

Convertimos las columnas de string a sus tipos correctos (int, double, etc.).

In [22]:
# Convertir múltiples columnas a sus tipos de datos correctos
# cast("int"): convierte a número entero
# cast("double"): convierte a número decimal (punto flotante)
# Encadenamos múltiples withColumn() para convertir varias columnas a la vez
df = df.withColumn("PassengerId", df.PassengerId.cast("int")) \
       .withColumn("Survived", df.Survived.cast("int")) \
       .withColumn("Pclass", df.Pclass.cast("int")) \
       .withColumn("Age", df.Age.cast("double")) \
       .withColumn("SibSp", df.SibSp.cast("int")) \
       .withColumn("Parch", df.Parch.cast("int")) \
       .withColumn("Fare", df.Fare.cast("double"))

# Ahora las columnas numéricas tienen el tipo correcto

In [23]:
# Verificar que los tipos de datos se hayan convertido correctamente
# Observa que PassengerId, Survived, Pclass, SibSp, Parch son 'integer'
# Observa que Age y Fare son 'double'
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



## 2.2 Cargar Archivo CSV con Formato (Forma Correcta)

**Sintaxis:** `spark.read.csv(ruta, sep="separador", inferSchema=True, header=True)`

Esta es la forma CORRECTA y más eficiente de cargar un CSV.

In [24]:
# Definir la ruta del archivo CSV
archivo = 'Titanic.csv'

# Cargar CSV con todas las opciones correctas:
# sep=',': el separador de columnas es la coma
# inferSchema=True: Spark detecta automáticamente los tipos de datos
# header=True: la primera fila contiene los nombres de las columnas
df = spark.read.csv(archivo, sep=',', inferSchema=True, header=True)

# Esta es la MEJOR PRÁCTICA: evita todo el trabajo manual anterior

In [25]:
# Mostrar el DataFrame cargado correctamente
# Observa que los nombres de columnas son correctos desde el inicio
df.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [26]:
# Verificar el esquema
# Observa que los tipos de datos ya son correctos automáticamente
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



## 2.3 Operaciones Básicas con DataFrames

Ahora veremos operaciones comunes para manipular DataFrames.

### Seleccionar Columnas Específicas

Usamos `select()` para elegir solo las columnas que necesitamos.

In [27]:
# Seleccionar solo las columnas Name, Sex y Age
# select() es una TRANSFORMACIÓN
# show(3): mostrar solo las primeras 3 filas
df.select("Name", "Sex", "Age").show(3)

+--------------------+------+----+
|                Name|   Sex| Age|
+--------------------+------+----+
|Braund, Mr. Owen ...|  male|22.0|
|Cumings, Mrs. Joh...|female|38.0|
|Heikkinen, Miss. ...|female|26.0|
+--------------------+------+----+
only showing top 3 rows



### Crear Campo Calculado en DataFrame

Podemos crear columnas calculadas basadas en otras columnas.

In [28]:
# Crear una columna calculada: Age * 11 (edad en meses aproximados)
# (df.Age * 11): multiplicar la columna Age por 11
# .alias("Meses"): dar un nombre a la nueva columna
# Esta columna solo existe en el resultado de select(), no modifica df
df.select("Name", "Sex", "Age", (df.Age * 11).alias("Meses")).show(4)

+--------------------+------+----+-----+
|                Name|   Sex| Age|Meses|
+--------------------+------+----+-----+
|Braund, Mr. Owen ...|  male|22.0|242.0|
|Cumings, Mrs. Joh...|female|38.0|418.0|
|Heikkinen, Miss. ...|female|26.0|286.0|
|Futrelle, Mrs. Ja...|female|35.0|385.0|
+--------------------+------+----+-----+
only showing top 4 rows



### Crear Nueva Columna en DataFrame

Usamos `withColumn()` para agregar permanentemente una columna al DataFrame.

In [29]:
# Crear una nueva columna permanente llamada "Meses"
# withColumn(nombre, expresión): agrega o reemplaza una columna
# df.Age * 12: calcular edad en meses
dfNuevo = df.withColumn("Meses", df.Age * 12)

# Mostrar el DataFrame con la nueva columna
# Ahora dfNuevo tiene una columna adicional llamada "Meses"
dfNuevo.show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+-----+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Meses|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+-----+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|264.0|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|456.0|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|312.0|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|420.0|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|420.0|
|          6|       0|     3|   

### Mostrar Valores Distintos de una Columna

Usamos `distinct()` para ver los valores únicos.

In [30]:
# Seleccionar la columna Pclass y mostrar solo valores únicos
# distinct(): elimina duplicados
# Resultado esperado: 1, 2, 3 (las tres clases del Titanic)
df.select("Pclass").distinct().show()

+------+
|Pclass|
+------+
|     1|
|     3|
|     2|
+------+



---
# PARTE 3: Configuración de Broadcast Join
---

Broadcast Join es una optimización que mejora el rendimiento de joins cuando uno de los DataFrames es pequeño.

## 3.1 Ver Tamaño de Broadcast Join

Spark tiene un umbral (threshold) que determina cuándo usar broadcast join automáticamente.

In [31]:
# Obtener el valor actual del umbral de broadcast join
# Este valor está en bytes
# Por defecto: 10 MB (10485760 bytes)
# Si un DataFrame es menor que este tamaño, Spark lo "transmite" a todos los nodos
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

'10485760b'

## 3.2 Modificar Tamaño de Broadcast Join

Podemos cambiar este umbral según nuestras necesidades.

In [32]:
# Cambiar el umbral de broadcast join a 100 MB (104857600 bytes)
# Esto permite que DataFrames más grandes se transmitan
# Útil cuando tienes suficiente memoria y quieres evitar shuffles costosos
Tamano = spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 104857600)

## 3.3 Ver Nuevo Tamaño de Broadcast Join

Verificamos que el cambio se haya aplicado correctamente.

In [33]:
# Obtener el nuevo valor del umbral
# int(): convertir de string a entero
Tamano = int(spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Mostrar el tamaño en MB (dividir entre 1024*1024)
print("El tamaño actual del Broadcast Join es:", Tamano / (1024 * 1024), "MB")

El tamaño actual del Broadcast Join es: 100.0 MB


## 3.4 Deshabilitar Broadcast Join

Podemos deshabilitar completamente el broadcast join automático.

In [34]:
# Establecer el umbral en -1 para deshabilitar broadcast join automático
# Útil si quieres controlar manualmente cuándo usar broadcast
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# Ahora Spark NO usará broadcast join automáticamente

---
# PARTE 4: Trabajar con DataFrames Empleado y Cargo
---

Ahora trabajaremos con dos DataFrames relacionados y aplicaremos broadcast join.

## 4.1 Crear DataFrames Empleado y Cargo

Cargamos dos archivos CSV que contienen información relacionada.

In [35]:
# Cargar archivo de Empleados
# Este archivo contiene información de empleados (ID, nombre, sueldo, etc.)
archivo = 'Empleado.csv'
Empleado = spark.read.csv(archivo, sep=';', inferSchema=True, header=True)

# NOTA: Asegúrate de tener este archivo en tu Google Drive o Local

In [36]:
# Cargar archivo de Cargos
# Este archivo contiene información de cargos/puestos (código, descripción)
# Típicamente es un DataFrame pequeño (tabla de referencia)
archivo = 'Cargo.csv'
Cargo = spark.read.csv(archivo, sep=';', inferSchema=True, header=True)

# NOTA: Asegúrate de tener este archivo en tu Google Drive o Local

In [37]:
# Mostrar ambos DataFrames para entender su estructura
# Empleado: DataFrame grande con datos de empleados
Empleado.show()

# Cargo: DataFrame pequeño con catálogo de cargos
Cargo.show()

+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+
|ID Empleado|   Nombre Empleado|          Estado|              Pais|Nacimiento|Género|Departamento|Cargo|       Nombre Jefe|
+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+
| 1102024149|    Spirea, Kelley|   California   |    Estados Unidos|  28-09-80|Female|           1|    2|      Elijiah Gray|
| 1001109612|   Darson, Jene'ya|   California   |    Estados Unidos|  10-05-83|Female|           1|    2|      Elijiah Gray|
| 1000974650|    Stanley, David|        Texas   |    Estados Unidos|  16-12-75|  Male|           1|    6|    Debra Houlihan|
| 1206043417|       Quinn, Sean|Massachusetts   |    Estados Unidos|  10-06-69|  Male|           1|    6|        Janet King|
| 1307060188| Boutwell, Bonalyn|   California   |    Estados Unidos|  02-04-72|Female|           1|    6|      Elijiah Gray|


## 4.2 Utilizar Broadcast Join

Aplicamos broadcast join para optimizar el join entre Empleado (grande) y Cargo (pequeño).

In [38]:
# Importar la función broadcast
from pyspark.sql.functions import broadcast

# Realizar un join optimizado con broadcast
# broadcast(Cargo): indica a Spark que transmita Cargo a todos los nodos
# Esto evita un shuffle costoso del DataFrame grande (Empleado)
# Empleado["Cargo"] == Cargo["Codigo"]: condición del join
dfTotal = Empleado.join(broadcast(Cargo), Empleado["Cargo"] == Cargo["Codigo"])

# Mostrar el resultado del join
# Ahora dfTotal tiene columnas de ambos DataFrames
dfTotal.show()

+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+
|ID Empleado|   Nombre Empleado|          Estado|              Pais|Nacimiento|Género|Departamento|Cargo|       Nombre Jefe|Codigo|            Cargo| Sueldo|
+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+
| 1102024149|    Spirea, Kelley|   California   |    Estados Unidos|  28-09-80|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000|
| 1001109612|   Darson, Jene'ya|   California   |    Estados Unidos|  10-05-83|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000|
| 1000974650|    Stanley, David|        Texas   |    Estados Unidos|  16-12-75|  Male|           1|    6|    Debra Houlihan|     6|          Manager|4000000|
| 1206043417|       Quinn, Sean|Massachusetts   |   

---
# PARTE 5: Expresiones SQL Avanzadas
---

Spark permite usar expresiones SQL directamente en las transformaciones de DataFrames.

## 5.1 Crear Nueva Columna con expr()

`expr()` permite escribir expresiones SQL como strings.

In [39]:
# Importar la función expr
from pyspark.sql.functions import expr

# Crear una nueva columna usando una expresión SQL con CASE WHEN
# CASE WHEN es como un if-else en SQL
# Si Sueldo > 3900000: 'Sueldo_alto'
# Si no: 'Sueldo_bajo'
cond = """case when Sueldo > 3900000 then 'Sueldo_alto'
               else 'Sueldo_bajo'
          end"""

# Aplicar la expresión para crear la columna "Rango_Sueldo"
dfTotal = dfTotal.withColumn("Rango_Sueldo", expr(cond))

# Mostrar el DataFrame con la nueva columna
dfTotal.show()

+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+
|ID Empleado|   Nombre Empleado|          Estado|              Pais|Nacimiento|Género|Departamento|Cargo|       Nombre Jefe|Codigo|            Cargo| Sueldo|Rango_Sueldo|
+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+
| 1102024149|    Spirea, Kelley|   California   |    Estados Unidos|  28-09-80|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|
| 1001109612|   Darson, Jene'ya|   California   |    Estados Unidos|  10-05-83|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|
| 1000974650|    Stanley, David|        Texas   |    Estados Unidos|  16-12-75|  Male|           1|    6|    Debra Houlihan|     6|          Mana

## 5.2 Crear Nueva Columna con selectExpr()

`selectExpr()` es similar a SELECT en SQL, permite expresiones complejas.

In [40]:
# Crear una expresión SQL más compleja con múltiples condiciones
# CASE WHEN anidado: evalúa múltiples condiciones en orden
cond = """case when Sueldo > 3900000 then 'Sueldo_alto'
               else case when Sueldo > 2000000 then 'Sueldo_medio'
                         else 'Sueldo_bajo'
                    end
          end as Rango_Sueldo2"""

# selectExpr permite seleccionar columnas Y aplicar expresiones SQL
# "*": seleccionar todas las columnas existentes
# cond: agregar la nueva columna calculada
dfTotal = dfTotal.selectExpr("*", cond)

# Mostrar el DataFrame con la nueva columna Rango_Sueldo2
dfTotal.show()

+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+
|ID Empleado|   Nombre Empleado|          Estado|              Pais|Nacimiento|Género|Departamento|Cargo|       Nombre Jefe|Codigo|            Cargo| Sueldo|Rango_Sueldo|Rango_Sueldo2|
+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+
| 1102024149|    Spirea, Kelley|   California   |    Estados Unidos|  28-09-80|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|  Sueldo_bajo|
| 1001109612|   Darson, Jene'ya|   California   |    Estados Unidos|  10-05-83|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|  Sueldo_bajo|
| 1000974650|    Stanley, David|        Texas   |    Estados Unidos|  16-12

## 5.3 Crear Nueva Columna con UDF (User Defined Function)

UDF permite crear funciones personalizadas en Python y usarlas en Spark.

In [41]:
# Importar todas las funciones de Spark
from pyspark.sql.functions import *
# Importar tipos de datos de Spark
from pyspark.sql.types import *

# Definir una función Python normal
# Esta función clasifica el salario en categorías
def detSalario(salario):
    if salario > 3900000:
        return "Sueldo_alto"
    elif salario > 2000000:
        return "Sueldo_medio"
    else:
        return "Sueldo_bajo"

# Convertir la función Python en una UDF de Spark
# udf(función, tipo_retorno)
# StringType(): la función retorna un string
udfDetSalario = udf(detSalario, StringType())

# Aplicar la UDF para crear una nueva columna
# udfDetSalario(dfTotal.Sueldo): aplicar la función a la columna Sueldo
dfTotal = dfTotal.withColumn("Rango_Sueldo3", udfDetSalario(dfTotal.Sueldo))

# Mostrar el DataFrame con la nueva columna creada por UDF
dfTotal.show()

+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+-------------+
|ID Empleado|   Nombre Empleado|          Estado|              Pais|Nacimiento|Género|Departamento|Cargo|       Nombre Jefe|Codigo|            Cargo| Sueldo|Rango_Sueldo|Rango_Sueldo2|Rango_Sueldo3|
+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+-------------+
| 1102024149|    Spirea, Kelley|   California   |    Estados Unidos|  28-09-80|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|  Sueldo_bajo|  Sueldo_bajo|
| 1001109612|   Darson, Jene'ya|   California   |    Estados Unidos|  10-05-83|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|  Sueldo_bajo|  Sueldo_bajo|
| 100

---
# PARTE 6: Cache y Persistencia de DataFrames
---

Cache y persistencia permiten reutilizar DataFrames sin recalcularlos.

## 6.1 Verificar si DataFrame está en Caché

Podemos verificar si un DataFrame está almacenado en memoria.

In [42]:
# Importar StorageLevel para trabajar con niveles de almacenamiento
from pyspark.storagelevel import StorageLevel

# Verificar si dfTotal está en caché
# is_cached: propiedad booleana (True/False)
# Resultado esperado: False (aún no lo hemos cacheado)
dfTotal.is_cached

False

## 6.2 Asignar DataFrame a Caché

Usamos `.cache()` para almacenar el DataFrame en memoria.

In [43]:
# Cachear el DataFrame en memoria
# cache() es equivalente a persist(StorageLevel.MEMORY_ONLY)
# El DataFrame se almacenará en memoria la primera vez que se ejecute una acción
dfTotal.cache()

# Verificar que ahora está marcado como cacheado
# Resultado esperado: True
dfTotal.is_cached

True

## 6.3 Asignar DataFrame a Persistencia

`persist()` ofrece más control sobre cómo se almacena el DataFrame.

In [44]:
# persist() sin argumentos usa el nivel por defecto (MEMORY_AND_DISK)
# Esto almacena el DataFrame en memoria, y si no cabe, usa disco
dfTotal.persist()

DataFrame[ID Empleado: int, Nombre Empleado: string, Estado: string, Pais: string, Nacimiento: string, Género: string, Departamento: int, Cargo: int, Nombre Jefe: string, Codigo: int, Cargo: string, Sueldo: int, Rango_Sueldo: string, Rango_Sueldo2: string, Rango_Sueldo3: string]

In [45]:
# Persistir con un nivel específico: MEMORY_ONLY
# StorageLevel.MEMORY_ONLY: solo en memoria (no usa disco)
# Si no cabe en memoria, las particiones se recalculan cuando se necesitan
dfTotal.persist(StorageLevel.MEMORY_ONLY)

# Ejecutar una acción para materializar la persistencia
# show() fuerza la evaluación y el almacenamiento en caché
dfTotal.show()

+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+-------------+
|ID Empleado|   Nombre Empleado|          Estado|              Pais|Nacimiento|Género|Departamento|Cargo|       Nombre Jefe|Codigo|            Cargo| Sueldo|Rango_Sueldo|Rango_Sueldo2|Rango_Sueldo3|
+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+-------------+
| 1102024149|    Spirea, Kelley|   California   |    Estados Unidos|  28-09-80|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|  Sueldo_bajo|  Sueldo_bajo|
| 1001109612|   Darson, Jene'ya|   California   |    Estados Unidos|  10-05-83|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|  Sueldo_bajo|  Sueldo_bajo|
| 100

In [46]:
# Liberar la caché actual
# unpersist() elimina el DataFrame de la memoria/disco
# Útil para liberar recursos cuando ya no necesitas el DataFrame cacheado
dfTotal = dfTotal.unpersist()

In [47]:
# Volver a persistir el DataFrame
# Esto es útil si quieres cambiar el nivel de persistencia
dfTotal = dfTotal.persist()

# Mostrar el DataFrame (materializa la persistencia)
dfTotal.show()

+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+-------------+
|ID Empleado|   Nombre Empleado|          Estado|              Pais|Nacimiento|Género|Departamento|Cargo|       Nombre Jefe|Codigo|            Cargo| Sueldo|Rango_Sueldo|Rango_Sueldo2|Rango_Sueldo3|
+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+-------------+
| 1102024149|    Spirea, Kelley|   California   |    Estados Unidos|  28-09-80|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|  Sueldo_bajo|  Sueldo_bajo|
| 1001109612|   Darson, Jene'ya|   California   |    Estados Unidos|  10-05-83|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|  Sueldo_bajo|  Sueldo_bajo|
| 100

## 6.4 Desasignar Persistencia del DataFrame

Liberamos completamente la memoria ocupada por el DataFrame.

In [48]:
# Liberar completamente la persistencia
# Después de esto, el DataFrame ya no estará en caché
dfTotal.unpersist()

DataFrame[ID Empleado: int, Nombre Empleado: string, Estado: string, Pais: string, Nacimiento: string, Género: string, Departamento: int, Cargo: int, Nombre Jefe: string, Codigo: int, Cargo: string, Sueldo: int, Rango_Sueldo: string, Rango_Sueldo2: string, Rango_Sueldo3: string]

In [49]:
# Verificar el nivel de almacenamiento actual
# rdd.getStorageLevel(): obtener información detallada del almacenamiento
# Resultado esperado: StorageLevel(False, False, False, False, 1)
# Esto significa que NO está en caché
dfTotal.rdd.getStorageLevel()

StorageLevel(False, False, False, False, 1)

---
# PARTE 7: Particionamiento de DataFrames
---

El particionamiento controla cómo se distribuyen los datos en el cluster.

## 7.1 Consultar Número de Particiones

Cada DataFrame está dividido en particiones que se procesan en paralelo.

In [50]:
# Obtener el número actual de particiones del DataFrame
# rdd: acceder al RDD subyacente del DataFrame
# getNumPartitions(): retorna el número de particiones
# Más particiones = más paralelismo (pero más overhead)
# Menos particiones = menos overhead (pero menos paralelismo)
dfTotal.rdd.getNumPartitions()

1

## 7.2 Incrementar Número de Particiones

Usamos `repartition()` para aumentar el paralelismo.

In [51]:
# Reparticionar el DataFrame a 6 particiones
# repartition(n): redistribuye los datos en n particiones
# IMPORTANTE: causa un SHUFFLE completo (costoso)
# Útil cuando: necesitas más paralelismo, datos están desbalanceados
newdf = dfTotal.repartition(6)

# Verificar el nuevo número de particiones
# Resultado esperado: 6
newdf.rdd.getNumPartitions()

6

## 7.3 Disminuir Número de Particiones

Usamos `coalesce()` para reducir particiones de forma eficiente.

In [52]:
# Reducir el número de particiones a 4
# coalesce(n): reduce a n particiones SIN shuffle completo
# Es MÁS EFICIENTE que repartition() para reducir particiones
# Útil cuando: has filtrado muchos datos y quieres menos particiones
newdf1 = newdf.coalesce(4)

# Verificar el nuevo número de particiones
# Resultado esperado: 4
newdf1.rdd.getNumPartitions()

4

---
# PARTE 8: Trabajar con JSON y Múltiples Archivos CSV
---

Spark puede leer diferentes formatos y unir múltiples archivos.

## 8.1 Leer Archivo JSON

Spark puede leer archivos JSON directamente.

In [53]:
# Leer archivo JSON con información de departamentos
# JSON: formato de intercambio de datos basado en texto
# Spark infiere automáticamente el esquema del JSON
archivo = 'Departamento.json'
Departamento = spark.read.option("multiline","true").json(archivo)

# Mostrar el DataFrame creado desde JSON
Departamento.show()

# NOTA: Asegúrate de tener este archivo en tu Google Drive

+------+--------------+
|Codigo|        Nombre|
+------+--------------+
|     1|          RRHH|
|     2|Administracion|
|     3|  Contabilidad|
|     4|     Logistica|
|     5|        Ventas|
|     6|   Informatica|
+------+--------------+



## 8.2 Unir Más de Dos Archivos CSV

Podemos cargar múltiples archivos CSV y unirlos en un solo DataFrame.

In [54]:
# Cargar archivo de ventas de Canadá
# Cada archivo contiene ventas de un país diferente
archivo = 'CA Sales.csv'
CA_Sales = spark.read.csv(archivo, sep=',', inferSchema=True, header=True)
CA_Sales.show()

# NOTA: Asegúrate de tener este archivo en tu Google Drive

+---------+---------+---+-----+--------+-------+
|ProductID|     Date|Zip|Units| Revenue|Country|
+---------+---------+---+-----+--------+-------+
|      725|1/15/1999|H1B|    1|115.4475| Canada|
|     2235|1/15/1999|H1B|    2| 131.145| Canada|
|      713|1/15/1999|H1B|    1|160.0725| Canada|
|      574| 6/5/2002|H1B|    1|869.1375| Canada|
|       94|2/15/1999|H1B|    1|  866.25| Canada|
|      609|2/15/1999|H1B|    1|778.8375| Canada|
|     2064|3/15/1999|H1B|    2| 976.395| Canada|
|      714|1/15/1999|H1B|    1|160.0725| Canada|
|      826|5/31/2002|H1B|    1|944.9475| Canada|
|     2149| 6/6/2002|H1B|    2| 871.395| Canada|
|      992|2/15/1999|H1B|    1|288.6975| Canada|
|      726|1/15/1999|M4X|    1|115.4475| Canada|
|      725|1/15/1999|M4X|    1|115.4475| Canada|
|      910|3/15/1999|M4X|    1|414.6975| Canada|
|      727|1/15/1999|R3T|    1| 62.9475| Canada|
|     1426|2/15/1999|R3T|    1|  286.02| Canada|
|     1182|1/15/1999|R3T|    1|157.4475| Canada|
|      976|5/31/2002

In [55]:
# Cargar archivo de ventas de Alemania
archivo = 'DE Sales.csv'
DE_Sales = spark.read.csv(archivo, sep=',', inferSchema=True, header=True)
DE_Sales.show()
# NOTA: Asegúrate de tener este archivo en tu Google Drive

+---------+---------+-----+-----+-------+-------+
|ProductID|     Date|  Zip|Units|Revenue|Country|
+---------+---------+-----+-----+-------+-------+
|      725|1/15/1999|41540|    1| 115.45|Germany|
|      787| 6/6/2002|41540|    1| 314.95|Germany|
|      788| 6/6/2002|41540|    1| 314.95|Germany|
|      940|1/15/1999|22587|    1|  687.7|Germany|
|      396|1/15/1999|22587|    1| 857.06|Germany|
|      734|4/10/2003|22587|    1|  330.7|Germany|
|      769|2/15/1999|22587|    1|  257.2|Germany|
|      499|1/15/1999|12555|    1|  846.3|Germany|
|     2254|1/15/1999|40217|    1|   57.7|Germany|
|       31|5/31/2002|40217|    1| 761.25|Germany|
|      475|2/15/1999|13583|    1|  970.2|Germany|
|      510|1/15/1999|22337|    1| 837.11|Germany|
|      499| 6/5/2002|22337|    1| 883.05|Germany|
|      289|2/15/1999|13587|    1| 865.99|Germany|
|      702|2/15/1999|13587|    1| 286.07|Germany|
|      910|3/15/1999|13587|    1|  414.7|Germany|
|      901|2/15/1999|13587|    2|  818.9|Germany|


In [56]:
# Cargar archivo de ventas de Francia
archivo = 'FR Sales.csv'
FR_Sales = spark.read.csv(archivo, sep=',', inferSchema=True, header=True)
FR_Sales.show()
# NOTA: Asegúrate de tener este archivo en tu Google Drive

+---------+---------+--------------+-----+-------+-------+
|ProductID|     Date|           Zip|Units|Revenue|Country|
+---------+---------+--------------+-----+-------+-------+
|      726|1/15/1999|75056 CEDEX 01|    1| 115.45| France|
|     1909|1/15/1999|75056 CEDEX 01|    2|  398.9| France|
|     1961|2/15/1999|75056 CEDEX 01|    1|  97.07| France|
|     1517|2/15/1999|75056 CEDEX 01|    1| 141.65| France|
|      606|2/15/1999|75056 CEDEX 01|    1| 314.74| France|
|     1518|2/15/1999|75056 CEDEX 01|    1| 141.65| France|
|      786|5/31/2002|         75001|    1|   68.2| France|
|      727|1/15/1999|75063 CEDEX 02|    2|  125.9| France|
|      559|1/15/1999|75063 CEDEX 02|    1| 585.64| France|
|      728|1/15/1999|75063 CEDEX 02|    2|  125.9| France|
|      964|4/10/2003|75063 CEDEX 02|    1|  320.2| France|
|      475|4/11/2003|75063 CEDEX 02|    1| 1037.4| France|
|     2055| 6/5/2002|75063 CEDEX 02|    1| 569.57| France|
|     1535|2/15/1999|75063 CEDEX 02|    1| 309.65| Franc

In [57]:
# Cargar archivo de ventas de México
archivo = 'MX Sales.csv'
MX_Sales = spark.read.csv(archivo, sep=',', inferSchema=True, header=True)
MX_Sales.show()
# NOTA: Asegúrate de tener este archivo en tu Google Drive

+---------+---------+----+-----+-------+-------+
|ProductID|     Date| Zip|Units|Revenue|Country|
+---------+---------+----+-----+-------+-------+
|     2235|1/15/1999|8650|    1|  65.57| Mexico|
|      837|2/15/1999|8650|    1| 839.95| Mexico|
|      491|2/15/1999|8650|    1| 815.33| Mexico|
|      426|5/31/2002|8650|    1| 843.15| Mexico|
|      400| 6/6/2002|8650|    1| 734.74| Mexico|
|     1131|2/15/1999|8650|    1| 419.95| Mexico|
|     2277|3/15/1999|8650|    1| 251.95| Mexico|
|      491|3/15/1999|8650|    1| 815.33| Mexico|
|      467| 6/8/2002|8650|    1| 900.64| Mexico|
|      565|4/12/2003|2000|    1| 890.66| Mexico|
|      604|4/12/2003|2000|    1| 494.81| Mexico|
|      940|3/15/1999|2000|    1|  687.7| Mexico|
|     2277|1/15/1999|8640|    1| 251.95| Mexico|
|      739|1/15/1999|2008|    1| 170.57| Mexico|
|     2188|1/15/1999|2008|    1| 233.57| Mexico|
|     1894|1/15/1999|2008|    1| 682.24| Mexico|
|      757|4/11/2003|2008|    1|  73.45| Mexico|
|      758|4/11/2003

In [58]:
# Unir todos los DataFrames de ventas en uno solo
# union(): combina filas de múltiples DataFrames con el mismo esquema
# Es como un UNION ALL en SQL (incluye duplicados)
# Encadenamos múltiples union() para combinar los 4 países
Sales = MX_Sales.union(FR_Sales).union(CA_Sales).union(DE_Sales)

# Mostrar el DataFrame unificado
Sales.show()

# Contar el total de registros
# Debería ser la suma de registros de los 4 países
Sales.count()

+---------+---------+----+-----+-------+-------+
|ProductID|     Date| Zip|Units|Revenue|Country|
+---------+---------+----+-----+-------+-------+
|     2235|1/15/1999|8650|    1|  65.57| Mexico|
|      837|2/15/1999|8650|    1| 839.95| Mexico|
|      491|2/15/1999|8650|    1| 815.33| Mexico|
|      426|5/31/2002|8650|    1| 843.15| Mexico|
|      400| 6/6/2002|8650|    1| 734.74| Mexico|
|     1131|2/15/1999|8650|    1| 419.95| Mexico|
|     2277|3/15/1999|8650|    1| 251.95| Mexico|
|      491|3/15/1999|8650|    1| 815.33| Mexico|
|      467| 6/8/2002|8650|    1| 900.64| Mexico|
|      565|4/12/2003|2000|    1| 890.66| Mexico|
|      604|4/12/2003|2000|    1| 494.81| Mexico|
|      940|3/15/1999|2000|    1|  687.7| Mexico|
|     2277|1/15/1999|8640|    1| 251.95| Mexico|
|      739|1/15/1999|2008|    1| 170.57| Mexico|
|     2188|1/15/1999|2008|    1| 233.57| Mexico|
|     1894|1/15/1999|2008|    1| 682.24| Mexico|
|      757|4/11/2003|2008|    1|  73.45| Mexico|
|      758|4/11/2003

841147

## 8.3 Join Final: Empleados con Departamentos

Realizamos un join entre dfTotal y Departamento.

In [59]:
# Realizar un join entre dfTotal (empleados) y Departamento
# dfTotal["Departamento"]: columna de departamento en dfTotal
# Departamento["Codigo"]: columna de código en Departamento
# Este es un INNER JOIN por defecto (solo filas que coinciden)
dfFinal = dfTotal.join(Departamento, dfTotal["Departamento"] == Departamento["Codigo"])

# Mostrar el DataFrame final con información completa
# Ahora tenemos: empleados + cargos + departamentos
dfFinal.show()

+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+-------------+------+------+
|ID Empleado|   Nombre Empleado|          Estado|              Pais|Nacimiento|Género|Departamento|Cargo|       Nombre Jefe|Codigo|            Cargo| Sueldo|Rango_Sueldo|Rango_Sueldo2|Rango_Sueldo3|Codigo|Nombre|
+-----------+------------------+----------------+------------------+----------+------+------------+-----+------------------+------+-----------------+-------+------------+-------------+-------------+------+------+
| 1102024149|    Spirea, Kelley|   California   |    Estados Unidos|  28-09-80|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II| 600000| Sueldo_bajo|  Sueldo_bajo|  Sueldo_bajo|     1|  RRHH|
| 1001109612|   Darson, Jene'ya|   California   |    Estados Unidos|  10-05-83|Female|           1|    2|      Elijiah Gray|     2|Admiiswtrativo II

---
# 🎉 ¡Felicitaciones!
---

Has completado todos los ejercicios prácticos de transformaciones avanzadas y optimización de DataFrames.

## 📚 Resumen de lo que Aprendiste:

1. ✅ **Configuración de Spark** en Google Colab
2. ✅ **Carga de datos** con y sin formato
3. ✅ **Transformaciones básicas**: renombrar, filtrar, convertir tipos
4. ✅ **Broadcast Join** para optimizar joins
5. ✅ **Expresiones SQL avanzadas**: expr(), selectExpr(), UDF
6. ✅ **Cache y persistencia** para reutilizar DataFrames
7. ✅ **Particionamiento** con repartition() y coalesce()
8. ✅ **Múltiples fuentes de datos**: CSV, JSON, union de archivos

---

## 🚀 Próximos Pasos:

- Aplica estos conceptos en el caso práctico del Titanic
- Experimenta con tus propios datasets
- Practica para la evaluación sumativa

---

## 💡 Recuerda:

- Siempre usa `header=True` e `inferSchema=True` al cargar CSVs
- Usa broadcast() para joins con DataFrames pequeños
- Cachea DataFrames que reutilizas múltiples veces
- Prefiere coalesce() sobre repartition() para reducir particiones

---

¡Éxito en tus análisis con Apache Spark! 🎯